# Corporate Reorganization — Retriever evaluation (SageMaker Processing)

This notebook runs evaluation on the **test** split via a SageMaker **Processing Job**.

It compares:
- the **fine-tuned** retriever (model artifact in S3)
- the **base** `answerdotai/ModernBERT-base`
- a **random baseline**

Outputs are written to S3 and include:
- `results.json`, `report.md`, `config.json`
- `runs/rankings.jsonl` (full ranked candidate list per query, for each system)


In [1]:
import os
import time
from pathlib import Path

import sagemaker
from dotenv import find_dotenv, load_dotenv
from sagemaker.huggingface import HuggingFaceProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

load_dotenv(find_dotenv(usecwd=True))

role = os.environ["SAGEMAKER_EXECUTION_ROLE_ARN"]
session = sagemaker.Session()

# Use the same bucket as training.
bucket = "sagemaker-us-east-1-371087393859"
prefix = "corporate_reorganization/retriever"

print("role:", role)
print("bucket:", bucket)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/xdg-ubuntu/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/lbrenap/.config/sagemaker/config.yaml


/home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


role: arn:aws:iam::371087393859:role/defaultrole
bucket: sagemaker-us-east-1-371087393859


In [2]:
processed_dir = Path("../data/final_annotations_gold/processed").resolve()
assert processed_dir.exists(), f"Missing processed_dir: {processed_dir}"

data_s3_uri = session.upload_data(
    path=str(processed_dir),
    bucket=bucket,
    key_prefix=f"{prefix}/data/processed",
)
inputs = {"data": data_s3_uri}

print("data_s3_uri:", data_s3_uri)


data_s3_uri: s3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/data/processed


In [7]:
fine_tuned_model_s3_uri = "s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-01-05-14-15-44-813/output/model.tar.gz"

timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
output_s3_uri = f"s3://{bucket}/{prefix}/eval_processing/{timestamp}"

processor = HuggingFaceProcessor(
    role=role,
    transformers_version="4.49.0",
    pytorch_version="2.5.1",
    py_version="py311",
    instance_type="ml.g5.12xlarge",
    instance_count=1,
)

processor.run(
    code="processing_eval/run_eval_sm.py",
    source_dir="../modernbert",
    inputs=[
        ProcessingInput(
            source=inputs["data"],
            destination="/opt/ml/processing/input/data",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/eval",
            destination=output_s3_uri,
        )
    ],
    arguments=[
        "--processed_dir",
        "/opt/ml/processing/input/data",
        "--output_dir",
        "/opt/ml/processing/output/eval",
        "--split",
        "test",
        "--fine_tuned_model_s3_uri",
        fine_tuned_model_s3_uri,
        "--base_model_name_or_path",
        "answerdotai/ModernBERT-base",
        "--k_values",
        "1,5,10,20",
        "--max_len_query",
        "4096",
        "--max_len_passage",
        "600",
        "--query_batch_size",
        "64",
        "--passage_batch_size",
        "256",
    ],
    wait=True,
    logs=True,
)

print("Evaluation outputs:", output_s3_uri)


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.processing:Uploaded ../modernbert to s3://sagemaker-us-east-1-371087393859/huggingface-2026-01-05-19-46-31-190/source/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-east-1-371087393859/huggingface-2026-01-05-19-46-31-190/source/runproc.sh
INFO:sagemaker:Creating processing-job with name huggingface-2026-01-05-19-46-31-190


...............................CodeArtifact repository not specified. Skipping login.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 96.1 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  DEPRECATION: Building 'deepspeed' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'deepspeed'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for deepspeed: filename=deepspeed-0.17.1-py3-none-any.whl size=1690871 sha256=d2d058d30f1661bf2bd6c941c9e56b37d7ba5e2128240472376f07cb977ccac8
  Stored in directory: /root/.cache/pip/wheels/34/86/36/22db26525829160fd1c4add33d8a834ec046b90abf45cd363b
Successf

In [ ]:
s3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/eval_processing/20260105_194631